In [5]:
import pandas as pd 

# Save to files
df_test= pd.read_csv('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/test_courses.csv', encoding='utf-8')
#df_all_courses.to_pickle('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/test_courses.pkl')

print(f"\nDataFrame shape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")
print("\nFirst row preview:")
print(df_test.head())


DataFrame shape: (29, 9)
Columns: ['title', 'credits', 'level', 'description', 'topics', 'learning_outcomes', 'prerequisites', 'examination_methods', 'course_code']

First row preview:
                                      title credits   level  \
0                          Systems Security  5 ECTS  Master   
1                       Smart Phone Sensing  5 ECTS  Master   
2  Research in Cyber Security – Hacking Lab    5 EC  Master   
3                    Error Correcting Codes    4 EC     MSc   
4                  Distributed Data Systems  5 ECTS     MSc   

                                         description  \
0  The course overviews principles, concepts, met...   
1  The course provides an introduction to the cur...   
2  The security of computer and telecommunication...   
3  Introduction into error-correcting codes; math...   
4  Starting in the mid-1990s, computing is underg...   

                                              topics  \
0  ['Hardware primitives', 'Operating syst

In [3]:
df_LU= pd.read_csv('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/all_mandatory_courses.csv', encoding='utf-8')

print(f"\nDataFrame shape: {df_LU.shape}")
print(f"Columns: {df_LU.columns.tolist()}")
print("\nFirst row preview:")
print(df_LU.iloc[0])


DataFrame shape: (175, 9)
Columns: ['title', 'credits', 'level', 'description', 'topics', 'learning_outcomes', 'prerequisites', 'examination_methods', 'course_code']

First row preview:
title                                        Calculus in One Variable B1
credits                                                      7.5 credits
level                                                                BSc
description            The aim of the course is to give a basic intro...
topics                 ['Number concept', 'Calculation with fractions...
learning_outcomes      ['within the framework of the course with conf...
prerequisites                                                         []
examination_methods    ['Written test comprising theory and problem s...
course_code                                                       FMAB65
Name: 0, dtype: object


In [4]:
# load RF 
import joblib 
RF = joblib.load('random_forest_model.pkl')

In [13]:
# Load sBERT tokenizer 
import nltk 
nltk.download('punkt') 
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize 
from sentence_transformers import SentenceTransformer 
import numpy as np 
import torch 
from utils.eval import * 
from utils.embedding import * 

model_name = "sentence-transformers/all-mpnet-base-v2"   # sBERT model
model = SentenceTransformer(model_name)
tokenizer = model.tokenizer
max_tokens = 256  # model limit

# compute cos_sim in learning objectives 
from sklearn.metrics.pairwise import cosine_similarity
import ast 
def embed_objectives(model, objectives):
    if len(objectives) == 0:
        return np.empty((0, model.get_sentence_embedding_dimension()))

    objectives = [normalize(obj) for obj in objectives]

    return model.encode(
        objectives,
        convert_to_numpy=True,
        normalize_embeddings=True   # important for cosine
    )

def extract_sim_matrix_neg(df_TUE, df_OVL, course1, course2, field='topics'):
    result = df_TUE.loc[df_TUE['course_code'] == course1, field] 
    LOs1 = ast.literal_eval(result.iloc[0])
    #LO2 can be either from original or overlap courses 
    if df_OVL.loc[df_OVL['course_code'] == course2, field].empty: 
        result = df_TUE.loc[df_TUE['course_code'] == course2, field] 
         
        if not result.iloc[0] == '[]': 
            LOs2 = ast.literal_eval(result.iloc[0])
        else: 
            return np.zeros((2,2))
    else: 
        result = df_OVL.loc[df_OVL['course_code'] == course2, field] 
        
        if not result.iloc[0] == '[]': 
            LOs2 = ast.literal_eval(result.iloc[0])
        else: 
            return np.zeros((2,2))
    # compute similarity 
    emb1 = embed_objectives(model, LOs1)
    emb2 = embed_objectives(model, LOs2)
    # Pairwise cosine similarity matrix
    sim_matrix = cosine_similarity(emb1, emb2)
    return sim_matrix
    #features.append(extract_features(sim_matrix, emb1, emb2))

def extract_sim_matrix_neg2(df_TUE, df_OVL, course1, course2, field='topics'):
    result = df_TUE.loc[df_TUE['course_code'] == course1, field] 
    LOs1 =result.iloc[0]
    #LO2 can be either from original or overlap courses 
    if df_OVL.loc[df_OVL['course_code'] == course2, field].empty:
        result = df_TUE.loc[df_TUE['course_code'] == course2, field] 
        if not result.empty: 
            LOs2 = result.iloc[0]
        else: 
            return np.zeros((2,2))
    else: 
        result = df_OVL.loc[df_OVL['course_code'] == course2, field] 
        if not result.empty: 
            LOs2 = result.iloc[0]
        else: 
            return np.zeros((2,2))
    # compute similarity 
    emb1 = embed_objectives(model, split_sentences(str(LOs1)))
    emb2 = embed_objectives(model,split_sentences(str(LOs2)))
    # Pairwise cosine similarity matrix
    sim_matrix = cosine_similarity(emb1, emb2)
    return sim_matrix
    #features.append(extract_features(sim_matrix, emb1, emb2))

def extract_features(S): #,embA,embB): 
    maxA = S.max(axis=1)
    maxB = S.max(axis=0)
    # Matching strength overall 
    features = [
        S.mean(),
        S.max(),
        S.std()
    ]
    # Coverage: how much of one course is covered by the other? 
    features.extend([
    (S > 0.5).mean(),
    #(maxB > 0.5).mean(),
    (S > 0.6).mean(),
    #(maxB > 0.6).mean(),
    (S > 0.7).mean(),
    ])
    return features 
 
from nltk.tokenize import PunktTokenizer
def split_sentences(text): 
    # Download the 'punkt' data if you haven't already
    # import nltk
    # nltk.download('punkt')
    
    #text = "Dr. Smith is here. She has a Ph.D. in Chemistry."
    tokenizer = PunktTokenizer()
    sentences = tokenizer.tokenize(text)
    
    return sentences
# Output: ['Dr. Smith is here.', 'She has a Ph.D. in Chemistry.']        

[nltk_data] Downloading package punkt to
[nltk_data]     /cephyr/users/nijdam/Alvis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /cephyr/users/nijdam/Alvis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /cephyr/users/nijdam/Alvis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /cephyr/users/nijdam/Alvis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [31]:
df_TUE = df_test 
df_OVL = df_LU 
field = 'learning_outcomes'

result = df_TUE.loc[df_TUE['course_code'] == course1, field] 
LOs1 =result.iloc[0]
#LO2 can be either from original or overlap courses 
if df_OVL.loc[df_OVL['course_code'] == course2, field].empty:
    result = df_TUE.loc[df_TUE['course_code'] == course2, field] 
    if not result.empty: 
        LOs2 = result.iloc[0]
    else: 
        print('none')
else: 
    result = df_OVL.loc[df_OVL['course_code'] == course2, field] 
    if not result.empty: 
        LOs2 = result.iloc[0]
    else: 
        print('none')
print(LOs1)
print(LOs2)

['Study security architecture of computer systems.', 'Analyse security issues of a specific operating system.', 'Develop skills in exploiting vulnerabilities of computer systems.', 'Develop counter measures against exploitation of computer systems.', 'Design secure computer systems.']
['demonstrate a knowledge and understanding of engineering including the scientific bases of engineering and knowledge about appropriate methodologies in the chosen field of the discipline', 'demonstrate an ability to critically, independently and creatively identify, formulate and handle relevant issues with a holistic view', 'demonstrate an ability to identify, formulate and solve a problem in the chosen field in an independent and creative manner within a given time period', 'be able to independently identify relevant sources of information, evaluate the relevance of this information and apply the correct conventions of documentation', 'demonstrate an ability to clearly present and discuss his/her conc

In [36]:
overlap_courses = [] 
overlap_course_codes = [] 
confidences = [] 
for course1 in df_test['course_code']: 
    print(course1) 
    overlap_course = [] 
    overlap_course_code = [] 
    confidence = []
    for i, course2 in enumerate(df_LU['course_code']):  #[0:23]:
        #print(course2)
        # we need to do all course_pairs 
        try: 
            sim_level = extract_sim_matrix_neg2(df_test, df_LU, course1, course2, field='level')
        except: 
            sim_level = [0]
        try: 
            sim_title = extract_sim_matrix_neg2(df_test, df_LU, course1, course2, field='title')
        except: 
            continue 
        try:
            sim_desc = extract_sim_matrix_neg2(df_test, df_LU, course1, course2, field='description')
        except: 
            continue 
        try: 
            sim_LOs = extract_sim_matrix_neg(df_test, df_LU, course1, course2, field='learning_outcomes') 
        except: 
            sim_LOs = np.zeros((2,2)) #[0,0,0,0,0,0]
        try: 
            sim_pre= extract_sim_matrix_neg2(df_test, df_LU, course1, course2, field='prerequisites')
        except: 
            sim_pre = [0]
        feature = [np.mean(sim_level)] + [np.mean(sim_title)] + extract_features(sim_desc) + extract_features(sim_LOs) + [np.mean(sim_pre)]
        # predict RF 
        feature = np.array(feature).reshape(1,-1)
        prediction = RF.predict(feature)[0]
        if prediction == 1: 
            #print(course2)
            confidence.append(RF.predict_proba(feature)[0][int(prediction)])
            overlap_course_code.append(course2) 
            overlap_course.append(df_LU['title'][i])
    if len(overlap_course) == 0: 
        print('no match found') 
    else: 
        print('matches: ',overlap_course)
        print('matches code: ',overlap_course_code)
        print('confidence: ',max(confidence))
        d = np.argmax(confidence) 
        print('best match: ',overlap_course[d])
        print('best match code: ',overlap_course_code[d])
    overlap_courses.append(overlap_course)
    overlap_course_codes.append(overlap_course_code)
    confidences.append(confidence) 

Studiegids — Study Guide _ Systems Security
matches:  ['Introduction to Programming', 'Design of Systems for Digital Transformation', 'Software Development in Teams - Project', 'Computer Security', 'Wireless Networks and Applications - System Design and Performance', 'Web Security', 'Usability Evaluation', 'Cryptography', 'Information Theory', 'Secure Systems Engineering', 'Advanced Web Security', 'Computer Architecture', 'Privacy Technologies', 'Operating Systems', 'Computer Organization / Datorteknik', 'Program Analysis', 'Real-Time Systems']
matches code:  ['EDAB05', 'EITA66', 'EDAF45', 'EITA25', 'ETSN11', 'EITF06', 'MAMF50', 'EDIN01', 'EITN45', 'EITP20', 'EITN41', 'EITF20', 'EITP55', 'EDAF35', 'EITF70', 'EDAP15', 'FRTN01']
confidence:  0.8335224368169974
best match:  Advanced Web Security
best match code:  EITN41
Studiegids — Study Guide _ Smart Phone Sensing
matches:  ['Advanced Interaction Design', 'Modern Wireless Systems - 5G and Beyond']
matches code:  ['MAMN01', 'EITP30']
con

In [37]:
# After your loop completes
# Create a DataFrame with the results
results_df = pd.DataFrame({
    'course_code': df_test['course_code'],  # Assuming this matches your loop order
    'overlap_courses': overlap_courses,
    'overlap_codes': overlap_course_codes,
    'confidences': confidences
})

# Save to CSV
results_df.to_csv('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/overlap_results_final.csv', index=False)